# Проверка подхода INN -> CDI_ID -> CFT_ID за февраль

Цель тетрадки:
1. Построить февральский периметр клиентов эквайринга по активным SA-договорам (без trx-условий).
2. Проверить цепочку маппинга `INN -> CDI_ID -> CFT_ID` без использования `agr_id`.
3. Сравнить результат с февральским Excel и посчитать coverage/расхождения.
4. Показать неоднозначности и примеры для анализа.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
report_month = '2026-02-01'
month_start = pd.to_datetime(report_month).strftime('%Y-%m-%d')
month_end = (pd.to_datetime(report_month) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
excel_inn_col = 'ИНН'
excel_header = 0

sa_type = 'SA'
mem_limit = '8g'
impala_db = 'sandbox_ai'
tez_queue = 'ai'
impala_user = 'Shestopalov-VYur'

chunk_size_inn = 800
chunk_size_cdi = 1200

output_dir = '/home/jovyan/documents/Equaring/Проверки/outputs/cft_mapping_validation_feb'

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r'[eE][+-]?\d+$', s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r'\.0+$', '', s)
    s = re.sub(r'\D', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s
    return s


def to_sql_in_list(values):
    out = []
    for v in values:
        s = str(v).replace("'", "''")
        out.append("'" + s + "'")
    return ', '.join(out)


def chunk_list(values, size):
    for i in range(0, len(values), size):
        yield values[i:i + size]


def fetch_in_chunks(imp, values, chunk_size, sql_builder):
    dfs = []
    for part in chunk_list(values, chunk_size):
        in_list = to_sql_in_list(part)
        if not in_list:
            continue
        with imp:
            imp.execute(f'set MEM_LIMIT={mem_limit}')
            d = imp.fetch(sql_builder(in_list))
        if len(d):
            dfs.append(d)
    return pd.concat(dfs, ignore_index=True) if len(dfs) else pd.DataFrame()

## 1) Excel-эталон за февраль: список INN

In [ ]:
excel_raw = pd.read_excel(excel_path, header=excel_header)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

if excel_inn_col not in excel_raw.columns:
    raise ValueError(f'Missing Excel INN column: {excel_inn_col}')

excel_inn_df = excel_raw[[excel_inn_col]].copy()
excel_inn_df.columns = ['inn_raw']
excel_inn_df['inn'] = excel_inn_df['inn_raw'].apply(normalize_numeric_str)
excel_inn_df = excel_inn_df[(excel_inn_df['inn'].notna()) & (excel_inn_df['inn'] != '')].copy()
excel_inn_df = excel_inn_df[['inn']].drop_duplicates().sort_values('inn').reset_index(drop=True)

excel_inn_list = excel_inn_df['inn'].tolist()
excel_inn_set = set(excel_inn_list)
print('excel_unique_inn =', len(excel_inn_set))
display(excel_inn_df.head(20))

## 2) Подключение к Impala (как в dashboard_mart_from_excel_feb.ipynb)

In [ ]:
if 'mem_limit' not in globals() or not mem_limit:
    mem_limit = '8g'
if 'impala_db' not in globals() or not impala_db:
    impala_db = 'sandbox_ai'
if 'tez_queue' not in globals() or not tez_queue:
    tez_queue = 'ai'
if 'impala_user' not in globals() or not impala_user:
    impala_user = 'Shestopalov-VYur'

imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': tez_queue},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': impala_user}
)
imp._init_connection()

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    imp.execute('select 1')

print(f'Impala initialized. user={impala_user}, db={impala_db}, queue={tez_queue}, mem_limit={mem_limit}')

## 3) Lake-периметр: активные SA-договоры на февраль (distinct INN)

In [ ]:
with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    lake_inn_df = imp.fetch(f"""
        select distinct
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where upper(trim(cast(a.acq_class as string))) = '{sa_type}'
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
    """)

lake_inn_df['inn'] = lake_inn_df['inn'].apply(normalize_numeric_str)
lake_inn_df = lake_inn_df[(lake_inn_df['inn'].notna()) & (lake_inn_df['inn'] != '')].drop_duplicates()
lake_inn_df = lake_inn_df.sort_values('inn').reset_index(drop=True)

lake_inn_list = lake_inn_df['inn'].tolist()
lake_inn_set = set(lake_inn_list)

print('lake_unique_inn =', len(lake_inn_set))
display(lake_inn_df.head(20))

## 4) Диагностика поля `id` в `ods.scd1_z_r2_ip_merchants` и проверка гипотезы `id == agr_id`

In [ ]:
with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')

    # 4.1 Базовая структура и примеры по id в r2-merchants
    r2_id_profile_df = imp.fetch("""
        select
            count(*) as rows_total,
            count(distinct cast(id as string)) as id_distinct_cnt,
            sum(case when id is null then 1 else 0 end) as id_null_cnt,
            sum(case when cast(id as string) rlike '^[0-9]+$' then 1 else 0 end) as id_numeric_rows
        from ods.scd1_z_r2_ip_merchants
    """)

    r2_id_examples_df = imp.fetch("""
        select
            cast(id as string) as id,
            cast(c_cl_org as string) as c_cl_org,
            cast(c_code_in_pr as string) as c_code_in_pr,
            cast(c_tariff_plan as string) as c_tariff_plan
        from ods.scd1_z_r2_ip_merchants
        where id is not null
        limit 20
    """)

    # 4.2 Сравнение id из r2 и abs_agr_id из agreements на SA-периметре февраля
    r2_vs_agr_cnt_df = imp.fetch(f"""
        with sa_agr as (
          select distinct cast(a.abs_agr_id as string) as agr_id
          from ods_alpha.scd1_agreements a
          join ods_alpha.scd1_companies c
            on c.n_cmp = a.n_cmp_client
          where upper(trim(cast(a.acq_class as string))) = '{sa_type}'
            and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
            and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
            and a.abs_agr_id is not null
            and coalesce(a.ods_deleted_flg, '0') <> '1'
            and coalesce(c.ods_deleted_flg, '0') <> '1'
        ),
        r2_ids as (
          select distinct cast(id as string) as id
          from ods.scd1_z_r2_ip_merchants
          where id is not null
        )
        select
          count(*) as sa_agr_id_cnt,
          sum(case when r2.id is not null then 1 else 0 end) as sa_agr_id_found_in_r2_id_cnt,
          sum(case when r2.id is null then 1 else 0 end) as sa_agr_id_not_found_in_r2_id_cnt,
          round(100.0 * sum(case when r2.id is not null then 1 else 0 end) / count(*), 2) as sa_agr_id_found_in_r2_id_pct
        from sa_agr a
        left join r2_ids r2 on r2.id = a.agr_id
    """)

    # 4.3 Примеры расхождений: agr_id из SA-февраля, которых нет среди r2.id
    r2_vs_agr_missing_examples_df = imp.fetch(f"""
        with sa_agr as (
          select distinct cast(a.abs_agr_id as string) as agr_id
          from ods_alpha.scd1_agreements a
          join ods_alpha.scd1_companies c
            on c.n_cmp = a.n_cmp_client
          where upper(trim(cast(a.acq_class as string))) = '{sa_type}'
            and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
            and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
            and a.abs_agr_id is not null
            and coalesce(a.ods_deleted_flg, '0') <> '1'
            and coalesce(c.ods_deleted_flg, '0') <> '1'
        ),
        r2_ids as (
          select distinct cast(id as string) as id
          from ods.scd1_z_r2_ip_merchants
          where id is not null
        )
        select a.agr_id
        from sa_agr a
        left join r2_ids r2 on r2.id = a.agr_id
        where r2.id is null
        limit 20
    """)

print('R2 id profile:')
display(r2_id_profile_df)

print('R2 id examples:')
display(r2_id_examples_df)

print('Hypothesis check (id == agr_id) on Feb SA perimeter:')
display(r2_vs_agr_cnt_df)

print('Examples of SA agr_id missing in r2.id:')
display(r2_vs_agr_missing_examples_df)

## 5) Маппинг INN -> CDI_ID

In [ ]:
def build_sql_inn_to_cdi(in_list):
    return f"""
    with ocrm as (
      select
        regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') as inn,
        cast(se.row_id as string) as row_id,
        row_number() over (
          partition by regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '')
          order by cast(se.last_upd as timestamp) desc,
                   cast(se.created as timestamp) desc,
                   cast(se.row_id as string) desc
        ) as rn
      from ocrm_ul.s_org_ext se
      where se.x_removed_flg = 'N'
        and se.x_duplicate_flg = 'N'
        and regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') in ({in_list})
    ),
    ocrm_latest as (
      select inn, row_id
      from ocrm
      where rn = 1
    ),
    cdi as (
      select distinct
        cast(cmo_ext_party_source_id as string) as row_id,
        cast(party_id as string) as cdi_id
      from cdiul.ext_id_org
      where upper(cmo_ext_source_system) like 'OCRM%'
    )
    select
      ol.inn,
      cdi.cdi_id
    from ocrm_latest ol
    left join cdi on cdi.row_id = ol.row_id
    """


inn_to_cdi_df = fetch_in_chunks(imp, lake_inn_list, chunk_size_inn, build_sql_inn_to_cdi) if len(lake_inn_list) else pd.DataFrame(columns=['inn', 'cdi_id'])
if len(inn_to_cdi_df):
    inn_to_cdi_df['inn'] = inn_to_cdi_df['inn'].apply(normalize_numeric_str)
    inn_to_cdi_df['cdi_id'] = inn_to_cdi_df['cdi_id'].astype(str)
    inn_to_cdi_df.loc[inn_to_cdi_df['cdi_id'].isin(['None', 'nan', '']), 'cdi_id'] = None
    inn_to_cdi_df = inn_to_cdi_df.drop_duplicates()

print('inn_to_cdi_rows =', len(inn_to_cdi_df))
display(inn_to_cdi_df.head(20))

## 6) Маппинг CDI_ID -> CFT_ID (`cmo_ext_source_system like 'CFT%'`)

In [ ]:
cdi_list = sorted(inn_to_cdi_df['cdi_id'].dropna().astype(str).unique().tolist()) if len(inn_to_cdi_df) else []

def build_sql_cdi_to_cft(in_list):
    return f"""
    select distinct
      cast(party_id as string) as cdi_id,
      cast(cmo_ext_party_source_id as string) as cft_id,
      cast(cmo_ext_source_system as string) as cft_source_system
    from cdiul.ext_id_org
    where cast(party_id as string) in ({in_list})
      and upper(cmo_ext_source_system) like 'CFT%'
    """


cdi_to_cft_df = fetch_in_chunks(imp, cdi_list, chunk_size_cdi, build_sql_cdi_to_cft) if len(cdi_list) else pd.DataFrame(columns=['cdi_id', 'cft_id', 'cft_source_system'])
if len(cdi_to_cft_df):
    cdi_to_cft_df['cdi_id'] = cdi_to_cft_df['cdi_id'].astype(str)
    cdi_to_cft_df['cft_id'] = cdi_to_cft_df['cft_id'].astype(str)
    cdi_to_cft_df.loc[cdi_to_cft_df['cft_id'].isin(['None', 'nan', '']), 'cft_id'] = None
    cdi_to_cft_df = cdi_to_cft_df.drop_duplicates()

print('cdi_to_cft_rows =', len(cdi_to_cft_df))
display(cdi_to_cft_df.head(20))

## 7) Финальный mapping-слой и QC неоднозначностей

In [ ]:
if len(lake_inn_df) == 0:
    mapping_df = pd.DataFrame(columns=['inn', 'cdi_id', 'cft_id', 'cft_source_system'])
else:
    mapping_df = lake_inn_df[['inn']].drop_duplicates().merge(inn_to_cdi_df, on='inn', how='left')
    mapping_df = mapping_df.merge(cdi_to_cft_df, on='cdi_id', how='left')

mapping_df = mapping_df.drop_duplicates()

inn_cdi_stat = (
    mapping_df.groupby('inn', dropna=False)['cdi_id']
    .nunique(dropna=True)
    .reset_index(name='cdi_cnt')
)
inn_cft_stat = (
    mapping_df.groupby('inn', dropna=False)['cft_id']
    .nunique(dropna=True)
    .reset_index(name='cft_cnt')
)
mapping_stat = inn_cdi_stat.merge(inn_cft_stat, on='inn', how='outer').fillna(0)

ambiguous_inn_cdi_df = mapping_stat[mapping_stat['cdi_cnt'] > 1].sort_values(['cdi_cnt', 'inn'], ascending=[False, True]).reset_index(drop=True)
ambiguous_inn_cft_df = mapping_stat[mapping_stat['cft_cnt'] > 1].sort_values(['cft_cnt', 'inn'], ascending=[False, True]).reset_index(drop=True)

print('mapping_rows =', len(mapping_df))
print('ambiguous_inn_to_cdi =', len(ambiguous_inn_cdi_df))
print('ambiguous_inn_to_cft =', len(ambiguous_inn_cft_df))
display(mapping_df.head(20))
display(ambiguous_inn_cdi_df.head(20))
display(ambiguous_inn_cft_df.head(20))

## 8) Сравнение с Excel + итоговые метрики + 5 примеров

In [ ]:
in_both_set = excel_inn_set & lake_inn_set
only_excel_set = excel_inn_set - lake_inn_set
only_lake_set = lake_inn_set - excel_inn_set

excel_map_stat = pd.DataFrame({'inn': sorted(excel_inn_set)}).merge(mapping_stat, on='inn', how='left')
excel_map_stat[['cdi_cnt', 'cft_cnt']] = excel_map_stat[['cdi_cnt', 'cft_cnt']].fillna(0)
excel_map_stat['has_cdi'] = excel_map_stat['cdi_cnt'] > 0
excel_map_stat['has_cft'] = excel_map_stat['cft_cnt'] > 0

excel_total = len(excel_map_stat)
excel_with_cdi = int(excel_map_stat['has_cdi'].sum())
excel_with_cft = int(excel_map_stat['has_cft'].sum())

summary_df = pd.DataFrame([
    {'metric': 'excel_unique_inn', 'value': len(excel_inn_set)},
    {'metric': 'lake_unique_inn_active_sa', 'value': len(lake_inn_set)},
    {'metric': 'inn_in_both', 'value': len(in_both_set)},
    {'metric': 'inn_only_in_excel', 'value': len(only_excel_set)},
    {'metric': 'inn_only_in_lake', 'value': len(only_lake_set)},
    {'metric': 'excel_inn_with_cdi', 'value': excel_with_cdi},
    {'metric': 'excel_inn_with_cft', 'value': excel_with_cft},
    {'metric': 'excel_cdi_coverage_pct', 'value': round(100.0 * excel_with_cdi / excel_total, 2) if excel_total else 0.0},
    {'metric': 'excel_cft_coverage_pct', 'value': round(100.0 * excel_with_cft / excel_total, 2) if excel_total else 0.0},
    {'metric': 'ambiguous_inn_to_cdi_cnt', 'value': len(ambiguous_inn_cdi_df)},
    {'metric': 'ambiguous_inn_to_cft_cnt', 'value': len(ambiguous_inn_cft_df)},
])

examples_only_excel_df = pd.DataFrame({'inn': sorted(list(only_excel_set))[:5]})
examples_only_lake_df = pd.DataFrame({'inn': sorted(list(only_lake_set))[:5]})
examples_missing_cdi_df = excel_map_stat[~excel_map_stat['has_cdi']][['inn']].sort_values('inn').head(5).reset_index(drop=True)
examples_missing_cft_df = excel_map_stat[~excel_map_stat['has_cft']][['inn']].sort_values('inn').head(5).reset_index(drop=True)

print('Summary:')
display(summary_df)

print('5 examples: only_excel')
display(examples_only_excel_df)

print('5 examples: only_lake')
display(examples_only_lake_df)

print('5 examples: missing CDI_ID in chain')
display(examples_missing_cdi_df)

print('5 examples: missing CFT_ID in chain')
display(examples_missing_cft_df)

## 9) Экспорт результатов

In [ ]:
out_dir = Path(output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

summary_path = out_dir / 'summary_feb_inn_cdi_cft.xlsx'
examples_path = out_dir / 'examples_feb_inn_cdi_cft.xlsx'
ambiguous_path = out_dir / 'ambiguous_feb_inn_cdi_cft.xlsx'
unmatched_path = out_dir / 'unmatched_feb_inn_cdi_cft.xlsx'
mapping_path = out_dir / 'mapping_feb_inn_cdi_cft.parquet'

summary_df.to_excel(summary_path, index=False)

examples_export_df = pd.concat([
    examples_only_excel_df.assign(example_type='only_excel'),
    examples_only_lake_df.assign(example_type='only_lake'),
    examples_missing_cdi_df.assign(example_type='missing_cdi'),
    examples_missing_cft_df.assign(example_type='missing_cft'),
], ignore_index=True)
examples_export_df.to_excel(examples_path, index=False)

with pd.ExcelWriter(ambiguous_path) as writer:
    ambiguous_inn_cdi_df.to_excel(writer, sheet_name='inn_to_cdi', index=False)
    ambiguous_inn_cft_df.to_excel(writer, sheet_name='inn_to_cft', index=False)

unmatched_export_df = pd.DataFrame({
    'inn': sorted(list(only_excel_set)) + sorted(list(only_lake_set)),
    'side': ['only_excel'] * len(only_excel_set) + ['only_lake'] * len(only_lake_set)
})
unmatched_export_df.to_excel(unmatched_path, index=False)

mapping_df.to_parquet(mapping_path, index=False)

print('Saved files:')
print(summary_path)
print(examples_path)
print(ambiguous_path)
print(unmatched_path)
print(mapping_path)

## 10) Быстрая ручная проверка: `agr_id == r2.id` для INN `7708044880`

In [ ]:
from rail_connectors.connection import connect

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

sql = """
with agr as (
    select distinct cast(a.abs_agr_id as string) as agr_id
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
    where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') = '7708044880'
      and upper(trim(cast(a.acq_class as string))) = 'SA'
      and a.abs_agr_id is not null
),
r2 as (
    select distinct cast(id as string) as r2_id
    from ods.scd1_z_r2_ip_merchants
    where id is not null
)
select
    a.agr_id,
    case when r.r2_id is not null then 1 else 0 end as found_in_r2
from agr a
left join r2 r on r.r2_id = a.agr_id
order by a.agr_id
"""

with imp:
    imp.execute("set MEM_LIMIT=8g")
    check_df = imp.fetch(sql)

check_df

## 11) Ручная проверка по `target_inn` в Озере (`agreements` + `r2_ip_merchants` + `r2_ip_merch_dog`)

In [ ]:
target_inn = '7708044880'
month_start = '2026-02-01'
month_end = '2026-02-29'

sql = f"""
with agr_feb as (
    select distinct
        cast(a.abs_agr_id as string) as agr_id
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') = '{target_inn}'
      and upper(trim(cast(a.acq_class as string))) = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and a.abs_agr_id is not null
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
),
r2 as (
    select
        cast(m.id as string) as r2_id,
        cast(m.c_cl_org as string) as c_cl_org,
        cast(m.c_code_in_pr as string) as c_code_in_pr,
        cast(m.c_tariff_plan as string) as c_tariff_plan
    from ods.scd1_z_r2_ip_merchants m
),
dog as (
    select
        cast(d.c_code_in_pr as string) as dog_code_in_pr,
        d.*
    from ods.scd1_z_r2_ip_merch_dog d
)
select
    a.agr_id,
    r2.r2_id,
    r2.c_cl_org,
    r2.c_code_in_pr,
    r2.c_tariff_plan,
    dog.*
from agr_feb a
left join r2
  on r2.r2_id = a.agr_id
left join dog
  on dog.dog_code_in_pr = r2.c_code_in_pr
order by a.agr_id
limit 500
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    target_inn_check_df = imp.fetch(sql)

target_inn_check_df